# 02 - OOP and Inheritance (Python)

This notebook builds the `Motor` / `TalonMotor` / `SparkMotor` hierarchy from `concept.md` idiomatically in Python. Read `concept.md` first, especially the polymorphism section. This notebook is the "how do I actually type it" half, and Python hides the pointer/reference bookkeeping that C++ needs entirely.

## The Base Class

Python doesn't have a built-in `interface` or `abstract class` keyword the way Java does, but the standard library's `abc` module ("abstract base classes") gets you the same guarantee: a subclass *must* implement any method marked `@abstractmethod`, or Python refuses to let you construct it.

In [1]:
from abc import ABC, abstractmethod

class Motor(ABC):
    def __init__(self, name):
        self.name = name
        self.current_power = 0.0

    @abstractmethod
    def set_power(self, power):
        """Every motor must define how it clamps and stores a requested power."""
        raise NotImplementedError

    def describe(self):
        return f"{self.name}: {self.current_power:.2f} power"

`set_power` has no real body: it exists purely to declare "every `Motor` must have this method." `describe` *does* have a real, usable default implementation, which subclasses are free to either use as-is or override with something more specific. That's the difference between a pure contract method and an inherited default.

## Subclasses That Override Behavior

`TalonMotor` and `SparkMotor` both inherit from `Motor`. Each implements `set_power` with its own clamp range, and each overrides `describe` to add a bit of hardware-specific detail.

In [2]:
class TalonMotor(Motor):
    MAX_POWER = 1.0

    def set_power(self, power):
        self.current_power = max(-self.MAX_POWER, min(self.MAX_POWER, power))

    def describe(self):
        return f"[Talon] {super().describe()}"


class SparkMotor(Motor):
    MAX_POWER = 0.8  # this particular Spark is geared for a mechanism with a lower safe limit

    def set_power(self, power):
        self.current_power = max(-self.MAX_POWER, min(self.MAX_POWER, power))

    def describe(self):
        return f"[Spark] {super().describe()}"

`super().describe()` calls the *base class's* version of `describe` from inside the override, so we reuse the shared formatting logic instead of duplicating it. Note also that constructing a bare `Motor()` directly would raise a `TypeError` — `abc` enforces that only concrete subclasses (ones that implement every abstract method) can actually be instantiated.

## Polymorphism

Now the payoff: a list holding a mix of `TalonMotor` and `SparkMotor` objects, processed by code that only ever mentions the shared base behavior (`set_power`, `describe`). Notice we request `1.5` power — out of range for *both* motors — and each object clamps it to its own limit, automatically, because each object runs its own overridden `set_power`.

In [3]:
motors = [TalonMotor("Left Drive"), SparkMotor("Intake Roller")]

for motor in motors:
    motor.set_power(1.5)  # deliberately out of range for both
    print(motor.describe())

[Talon] Left Drive: 1.00 power
[Spark] Intake Roller: 0.80 power


The loop body never checks "is this a `TalonMotor` or a `SparkMotor`?". It doesn't need to. Each object already knows how to handle `set_power` and `describe` correctly for itself. This is exactly the property that lets a `Subsystem` be written once against `Motor` in general and later be handed any concrete motor type without modification.

## Try It Yourself

No solutions are provided — these are meant to be worked through on your own or with a mentor or another student.

1. Add a third subclass, `VictorMotor`, with its own `MAX_POWER` and its own `describe` override, and add an instance of it to the `motors` list.
2. Add a new abstract method to `Motor` called `stop()` that every subclass must implement (perhaps just calling `set_power(0)`), and implement it in all three subclasses.
3. Try instantiating `Motor("test")` directly in a new cell and read the error Python gives you. What does it tell you about *why* `Motor` can't be constructed on its own?

In [4]:
# Your code here
